# Inheritance

<img src="/introduction-to-python-programming/images/09/inheritance.png" alt="hierarchy of cars" width="400px" />

_Inheritance_ is a central principle in OOP which leverages commonalities and differences between objects which have to be dealt with in our code. The big advantage, as we will see, is that inheritance minimizes _redundancy_ (we do not have to write the same code over and over again) and thus also facilitates maintenance (if we need to change something in the code, we need to do it only once, if the functionality is shared by different classes). This may sound very abstract, so let us have a look at a concrete example.

## Example

Assume our bank offers two different kind of accounts:

* __Savings Account__: We record the account number, holder and balance with each account. The balance has to be >=0. We can apply an interest rate which is defined once for all savings accounts. Money can be deposited into the account. The account statement that can be printed includes the account number, holder and balance.
* __Checking Account__: We record the account number, holder and balance with each account. The balance has to be greater than or equal to a credit range which is determined on a per-customer basis. For instance, if the credit range determined for Marc is &euro;500, his balance may not be less than - &euro;500. Money can be deposited into the account. The account statement that can be printed includes the account number, holder and balance.

Usually, the development of an application starts with a description of some real-world objects (such as the accounts) like above. Now, we have to map these descriptions into an object-oriented design. We can make the following statements about our application setting:

1. Both the savings account and the checking account are some type of account.
2. We can deposit money into either account type.
3. The account statements to be printed out are the same for the two account types.
4. The savings account and the checking account both have the following attributes:
   * account number
   * account holder
   * balance
5. The savings accounts have an additional parameter, the interest rate, but which is the
same for all savings accounts.
6. Savings accounts have a behavior that checking accounts don’t have: We can apply
the interest rate on their balances.
7. Each checking account has its own particular credit range.
8. Cash withdrawal works differently for the two account types. In the savings account,
the balance may not be less than 0, while for checking accounts, the balance may not
be less than the credit range deﬁned for the particular client.

## Base classes provide general functionality

All of the above considerations will be reﬂected in our class design. First of all, we will code a _base class_ which implements all the things that are shared by the two types of accounts: the attributes holder, number and balance, methods for deposit and withdrawal as well as for printing out the state of the account. The following listing shows our base class, `Account`, which doesn’t look much different from the previous class. In fact, we could instantiate objects directly from this class and use them as accounts.

In [ ]:
class Account:
    # general functionality for accounts
    def __init__(self, iban, person):
        self.balance = 0
        self.iban = iban
        self.holder = person

    def deposit(self, amount):
        if amount > 0:
            self.balance += amount

    def __str__(self) -> str:
        return str(type(self)) + ": " + self.holder + " " + \
             self.iban + "\n" + "balance=" + str(self.balance)

## Derived classes for specific purposes
Let us turn to the savings accounts. In fact, they are a specialized case of the `Account` class we have just written. Next, we are going to write a class called `SavingsAccount` which _extends_ the `Account` class, or is _derived_ from it. This basically means that all functionality that is available in the `Account` class is also available in the `SavingsAccount` class. We tell Python that the `SavingsAccount` class is _based_ on the `Account` class by starting the class deﬁnition with class `SavingsAccount(Account)`. We say that a _derived class/subclass_ is _based_ on a _superclass/base class_. This means that anything that is available in the superclass is also available in the subclass.


In [ ]:
# Special Cases are implemented in Derived Classes / Subclasses
class SavingsAccount(Account): # extends the Account class
    ''' class for objects representing savings accounts.
    shows how a class can be extended. '''
    interest_rate = 0.035
    # METHODS
    def apply_interest(self):
        self.balance *= (1+SavingsAccount.interest_rate)

`SavingsAccount` provides some additional functionality (applying an interest rate). In our main application, we instantiate two savings accounts.

In [ ]:
annesAcc = SavingsAccount("DE42", "Anne")
stefansAcc = SavingsAccount("DE38", "Stefan")
annesAcc.deposit(500)
annesAcc.apply_interest()
print(annesAcc)

This is how our class hierarchy and our instantiated objects look like at this point:

<img src="/introduction-to-python-programming/images/09/inheritance1.png" alt="image depicting how the classes and objects relate to each other" width="200px"/>

Note that although the instance attributes `balance`, `iban` and `holder` were set in the constructor of the `Account` class, they are attributes of the instance objects. We set them as `self.balance`, not as `Account.balance` in the constructor method. When we apply any instance method on an object, the class hierarchy is searched from bottom to the top. For instance, we did not deﬁne a constructor method in the `SavingsAccount` class. Because the `SavingsAccount` class is based on the `Account` class, Python continues looking for an init method at the superclass (`Account`), ﬁnds it there and executes this method when creating a new object by calling the `SavingsAccount` class.

When calling `annesAcc.deposit(200)`, Python starts looking for a `deposit` method at the `annesAcc` object, but does not ﬁnd one there. It then looks at the class from which the object was created, which happens to be `SavingsAccount`. It does not ﬁnd the method there either, so it continues looking for the class in the superclass of
`SavingsAccount`, `Account`, where it ﬁnally ﬁnds the method. The `apply_interest(self)` method is found at the `SavingsAccount` class, which means that the search stops here and executes this method.

## Subclasses can override the behavior of their superclasses

Now, we deﬁne a class called `CheckingAccount` which is also based on the `Account` class. However, the general `withdraw(self, amount)` method, which we deﬁned in the `Account` class and which does not allow a withdrawal to result in a negative
balance, does not apply to checking accounts. Recall that a checking account may have a negative balance, but only up to a credit range which is deﬁned on a per-customer basis. The following listing shows how we can redeﬁne/replace, or (as in OOP-speak) _override_ methods of a superclass in a subclass.

In [ ]:
# Inheritance and Initialization

class Account:

    def __init__(self, iban, person):
        self.balance = 0
        self.iban = iban
        self.holder = person
    
    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if amount < 0:
            print("Withdrawing negative amounts is not possible!")
            return 0
        self.balance -= amount
        return amount


class CheckingAccount(Account):

    def __init__(self, iban, person, credit_range):
        Account.__init__(self, iban, person)
        # super().__init__(self, num, person) # try out both versions
        self.credit_range = credit_range

    def withdraw(self, amount): # this function overrides the withdraw method of the superclass Account
        amount = min(amount, abs(self.balance + self.credit_range))
        cash = super().withdraw(amount) # self included!
        return cash


annesAcc = CheckingAccount("DE42", "Anne", 1000)
# Which methods get called during initialization?

# Your code:
# 1. Deposit 200 Euros.
# 2. Withdraw 700 Euros.
# Which methods get called in case case?

As we can see, the `CheckingAccount` class provides a constructor method (`__init__). When creating an object by`calling the `CheckingAccount` class, Python starts looking for a constructor at this class and immediately ﬁnds one.
It executes the `__init__` method of the `CheckingAccount` class and sets the four attributes of the object being created.
The constructor method of the superclass `Account` is not executed in this case. The `CheckingAccount` class also overrides the `withdraw(self, amount)` method of its superclass. _Overriding_ a method simply works by giving the exact same name to a method. When we call this method on the `annesAcc` object, Python again starts looking for the method in the object and then at the classes it is linked to from bottom to top. It finds the method at the `CheckingAccount` class and executes
this method.

<img src="/introduction-to-python-programming/images/09/inheritance2.png" alt="illustration showing that CheckingAccount is a subclass of Account" width="200px" />

&#128527; _Which methods of which class are called in each line?_

```
annesAcc = CheckingAccount("DE42", "Anne", 500)
annesAcc.deposit(200)
annesAcc.withdraw(350)
print(annesAcc)
```

## Methods with the same name can do different things

The image below shows the complete class hierarchy and our two account objects.
When creating `annesAcc`, Python looks for the constructor method in the `SavingsAccount` class, does not find one, and moves on to the `Account` class. It then modiﬁes the newly created object as coded in the init method of the `Account` class. When creating the `stefansAcc` object, Python ﬁnds a constructor method in the `CheckingAccount` class and uses this constructor directly to modify the newly created object. When we apply the `deposit` method on either the `annesAcc` or the stefansAcc object, Python will execute the `deposit` method of the `Account` class, because it does not find a `deposit` method anywhere lower in the tree.

<img src="/introduction-to-python-programming/images/09/inheritance3.png" alt="drawing of complete class hierarchy corresponding to code above" width="300px">

When we apply the withdraw method on the `stefansAcc` object, again, Python finds the method deﬁnition in the `CheckingAccount` class and executes this method.
When we apply `withdraw` on `annesAcc`, Python executes the `deposit` method defined in the `Account` class.


&#128527; Which methods are executed when calling them on `annesAcc` or `stefansAcc`?
* `__init__`
* `__str__`
* `deposit`
* `withdraw`
* `apply_interest`

## Duck typing
We just saw an example of _polymorphism_. The word _polymorphism_ is derived from the Greek word for "having multiple forms". In this case, it means that we can call a method that has the same name (for instance withdraw) on several objects (here `annesAcc` and `stefansAcc`), and what happens depends on the inheritance hierarchy of the classes from which the objects were created. The nice thing here is, at the point in time when we do the withdrawal, we simply say `accountObject.withdraw(amount)` and we do not need to care about whether the `accountObject` is a savings or a checking account. Python knows which inheritance lines to follow and the desired behavior is produced in either case.
Of course, we could simply have deﬁned the `withdraw` method twice, once in the `SavingsAccount` class and once in the `CheckingAccount` class. We deﬁned the method in the superclass here in order to give an example for overriding methods.
This might even make sense if our bank provides many more types of accounts and in most of these accounts, the withdrawal works as deﬁned in the `Account` class. Then, most of the classes derived from `Account` simply default to this behavior, while we can provide special behavior for checking accounts.

<div class="alert alert-block alert-info">

<img src="/introduction-to-python-programming/images/09/duck.png" alt="image of a duck" width="150px"/>

__Duck Typing__: If it walks like a duck and it quacks like a
duck, then it must be a duck.

When calling a method on objects of different classes, as long as a method of that name is defined for the class or a superclass, we can view these objects as similar, since they can perform the expected behavior.
Of course, the methods may be implemented slightly different, so what really happens depends on the class hierarchy from which the objects were created.
</div>

## UML diagram with inheritance
For the sake of completeness, this is how the UML diagram looks like in this case:

<img src="/introduction-to-python-programming/images/09/uml_inheritance.png" alt="UML diagram with inheritance signaled by triangle" width="500px" />

In the image: _number = iban_

In UML class diagrams, inheritance is shown by connecting the classes with a line. The triangle points to the superclass. It is convention to put the superclass on top of its subclasses (if space allows). Class attributes and static methods are underlined in UML class diagrams (such as the class attribute `interest_rate` in the example).


## Initializing subclasses

We said that OOP is great because it helps us to minimize redundancy and thus makes code maintance easier. Minimizing redundancy means to never write the same piece of code twice. Compare the constructor methods of our `Account` class and the
`CheckingAccount` class, which is derived from it. Here, we just copied and pasted parts of what is going on inside the constructor method of Account into the constructor method of `CheckingAccount`.

In [ ]:
class Account:
    ''' a class representing an account '''
    # CONSTRUCTOR
    def __init__(self, iban, person):
        self.balance = 0
        self.iban = iban
        self.holder = person
    # METHODS
    # ...
        
class CheckingAccount(Account):
    ''' class for objects representing checking accounts.
        shows how methods can be overridden '''
    # CONSTRUCTOR
    def __init__(self, iban, person, credit_range):
        self.iban = iban
        self.holder = person
        self.balance = 0
        self.credit_range = credit_range
    # METHODS
    # ...

As OOP is great, there is of course a better way to do it. We can call the constructor
of a superclass in the constructor of a subclass. In this particular case, it looks like this:

In [ ]:
class CheckingAccount(Account):
    ''' class for objects representing checking accounts.
        shows how methods can be overridden '''
    # CONSTRUCTOR
    def __init__(self, iban, person, credit_range):
        Account.__init__(self, iban, person)
        self.credit_range = credit_range
    # METHODS
    # ...

The line `Account.__init__(self, iban, person)` calls the constructor method of the `Account` class. Here, we need to
explicitly pass the `self` parameter, because we are not calling the method "on an object", but "on the class". By passing  the `self` parameter, Python will know which object to operate on when it executes the constructor method of the `Account` class.
In general, we can call any method of a superclass in this way, not only the constructor method. For instance, we could implement the withdraw method in a very general way in the base class, and do all the consistency checking in the subclasses, as shown in the following example. (In this case, we are not really saving lines of code, but in a more complex case, the method in the superclass might be more complex, and then it really makes sense to structure your code like this.)

In [ ]:
class Account:
    def withdraw(self, amount):
        self.balance -= amount

class SavingsAccount(Account):
    # METHODS
    def withdraw(self, amount):
        ''' balance must be > 0 '''
        if amount > self.balance:
            amount = self.balance
        cash = Account.withdraw(self, amount)
        return cash

class CheckingAccount(Account):
    ''' class for objects representing checking accounts.
        shows how methods can be overridden '''
    # CONSTRUCTOR
    def __init__(self, num, person, credit_range):
        Account.__init__(self, num, person)
        self.credit_range = credit_range
    # METHODS
    def withdraw(self, amount):
        ''' balance must be > credit range '''
        amount = min(amount,
             abs(self.balance + self.credit_range))
        cash = Account.withdraw(self, amount)
        return cash

Finally (perhaps the best way to pass on initialization calls in most cases), instead of referencing the superclass explicitly, we can use `super()`, which returns a _proxy object_ that allows us to access methods of the base class. This always calls the class closest in the inheritance hierarchy, i.e., it looks for the next `__init__` method according to the _Method Resolution Order_ (explained in detail below). It works like an instance call, i.e., it does not require explicitly handing over `self`.

In [ ]:
class CheckingAccount(Account):
  def __init__(self, num, person, credit_range):
    super().__init__(num, person)
    self.credit_range = credit_range
    
  def withdraw(self, amount):
    amount = min(amount, abs(self.balance + self.credit_range))
    cash = super().withdraw(amount) # self included!
    return cash

## Multiple Inheritance

In some OOP language (e.g. Java) classes can only extend a single class. In Python, a class can inherit from more than one class.

```
class SomeClass(FirstParent, SecondParent, ...):
   # ...
```

When resolving method calls (including `__init__(self,...)` and `super()`), Python looks in the parent classes from left to right, first in `FirstParent`, if not found in `SecondParent` and so on.

In [ ]:
# Multiple Inheritance
class FirstParent:
    @staticmethod
    def firstMethod():
        return "Hello"

class SecondParent:
    @staticmethod
    def secondMethod():
        return  "world"

class ChildClass(FirstParent, SecondParent):
    def __init__(self) -> None:
        super().__init__()
        print(super())

    def someMethod(self):
        print(super())
        print(super().firstMethod() + " " + super().secondMethod())

__Method Resolution Order__

= The order in which Python checks for the methods.

In [ ]:
# View the Method Resolution Order
ChildClass.__mro__

In [ ]:
c = ChildClass()
print(c.firstMethod)
print(c.secondMethod)
c.someMethod()

So far, so good. Can you predict what happens if we add a `secondMethod` to `FirstParent?`

In [ ]:
# Multiple Inheritance
class FirstParent:
    @staticmethod
    def firstMethod():
        return "Hello"
    @staticmethod
    def secondMethod():
        return  "universe"

class SecondParent:
    @staticmethod
    def secondMethod():
        return  "world"

class ChildClass(FirstParent, SecondParent):
    def __init__(self) -> None:
        super().__init__()
        print(super())

    def someMethod(self):
        print(super())
        print(super().firstMethod() + " " + super().secondMethod())

c = ChildClass()
print(c.secondMethod)
c.someMethod()

## Everything in Python is an object

Lists and dictionaries come with a special syntax for creation.

In [ ]:
# create a new list object
myList = []
# call a method of the list object
myList.append(4)
# create a new dictionary object
myDict = {}
# call a method of the dictionary object
myDict["someKey"] = "someValue"

Line 8 calls a hook method of the dictionary:

`__setitem__(self, key, value)`

 We can even subclass the built-in classes, e.g., by overriding hooks:

In [ ]:
class TalkingDict(dict):
    # Constructor
    def __init__(self):
        print("Starting to create a new dictionary...")
        dict.__init__(self)
        print("Done!")
    # Methods
    def __setitem__(self, key, value):
        print("Setting", key, "to", value)
        dict.__setitem__(self, key, value)
        print("Done!")

print("We are going to create a talking dictionary!")
myDict = TalkingDict()
myDict["x"] = 42

## Summary

### Understanding OOP is useful because...
* In real life, you will rarely program _from scratch_.
* You will extend / customize other people's code.
* Frameworks are collections of superclasses that implement common programming tasks.
      In this course, we will get more programming experience in Python by exploring some broadly used frameworks.
* You need to understand how the classes of the framework work together.
* You write subclasses that specialize the behavior for your task.
* Recipes for how to extend superclasses in an effective way are called __Design Patterns__.

### A few remarks on terminology

For beginners, OOP may seem like extra coding efforts that are not necessary in some cases. So why is it essential to understand OOP if you want to be a good programmer these days? The main reason is that you will rarely program from scratch, but by using, extending and customizing other people’s code. Particularly, you will use frameworks
which are basically collections of superclasses that implement common programming tasks. Often, you will just have to write a subclass for one of the framework’s classes that specializes the behavior for you specific application. In fact, extensions in this way are so common that design patterns have been developed that are recipes for good
ways to extend or leverage classes.
Another issue of confusion is the term _data encapsulation_, because it is used for two aspects of OOP. The ﬁrst, which we introduced in this tutorial, is that data should be hidden, i.e. only accessed via instance methods, in order to make sure that our object is in a valid state. The other aspect of OOP that is often described using the term
encapsulation is that OOP helps to wrap up program logic behind interfaces (≈ class names and the methods of the functions) such that each functionality is only defined once in a program. When you work with the functionality, you just call the respective class’s method and you don’t care what happens "under the hood". In fact, you can even change the implementations that happen "under the hood" without changing the code that uses the interface. This can be useful for instance if you suddenly think of a more efﬁcient implementation of a particular function. To monkey-patch the code (not always recommended &#x1F913; but technically possible), all you need to change is
exactly this function.


&#127881; Congratulations! You have mastered several advanced topics related to object-oriented programming in Python!

## References

Mark Lutz: _Learning Python_, Part VI, 4th edition, O'Reilly, 2009.
  
Michael Dawson: _Python Programming for the Absolute Beginner_, Chapters 8 and 9, 3rd edition, Course Technology PTR, 2010.
  